<!-- dads-lab-header -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M03/M03_Lab1_Prompting_Techniques.ipynb)

![M03 Lab1 Prompting Techniques](https://raw.githubusercontent.com/mdehghani86/DADS5250-GenAI/main/labs/M03/assets/images/M03_Lab1_Prompting_Techniques_banner.png)

In [ ]:
# === Shared lab setup: install dads5250 + load API key + sticky pill ===
# Installs the shared utilities (pp, pretty_print, lab_pill, model constants,
# setup_openai, setup_gemini) once per Colab runtime. The same OPENAI_API_KEY
# / GEMINI_API_KEY Colab secrets are used across every DADS 5250 lab — set
# them once in the 🔑 sidebar and they're picked up automatically.
import os
import importlib.util
if importlib.util.find_spec("dads5250") is None:
    !pip install -q "git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils"

from dads5250 import (
    pp,
    pretty_print,
    lab_pill,
    setup_openai,
    setup_gemini,
    DEFAULT_CHAT_MODEL,   # newest reasoning model that supports temperature
    DEFAULT_MINI_MODEL,   # newest mini model that supports temperature
    DEFAULT_EMBED_MODEL,  # current embeddings default
    DEFAULT_GEMINI_MODEL, # tracks the latest stable flash
)

lab_pill('M03 Lab 1 — Prompting Techniques')            # sticky banner so you always see which lab you're in


## API check

Confirm the API connection before we start. Your key is read from a Colab Secret, an environment variable, or a hidden prompt if neither is set.

In [ ]:
# === API check: confirm the connection and show the model(s) this lab uses ===
client = setup_openai()        # loads OPENAI_API_KEY + verifies it works

pp({
    "OpenAI":      "connected",
    "chat model":  DEFAULT_CHAT_MODEL,
    "mini model":  DEFAULT_MINI_MODEL,
}, title="API check")

<a href="https://colab.research.google.com/github/mdehghani86/AppliedGenAI/blob/main/M3_Lab1_Prompting_Strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- Intro Section -->
<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 30px; border-radius: 12px; text-align: center; box-shadow: 0 4px 12px rgba(0,0,0,0.1);">
    <h1 style="margin-bottom: 10px; font-size: 32px;">Introduction to Prompting Strategies</h1>
    <p style="font-size: 18px; margin: 0;">Instructor: <strong>Dr. Dehghani</strong></p>
</div>

<!-- Spacer -->
<div style="height: 30px;"></div>

<!-- Why It Matters Section -->
<div style="background: #ffffff; padding: 25px; border-radius: 10px; border-left: 6px solid #0055d4; box-shadow: 0 4px 8px rgba(0,0,0,0.05);">
    <h2 style="margin-top: 0; color: #001a70;">Why Prompting Strategies Matter</h2>
    <p style="font-size: 16px; line-height: 1.6;">
        Imagine you’re working with a junior engineer. You say:  
        <em>“Optimize the system.”</em><br>
        They’ll probably ask: <em>“Which system? Optimize for cost, speed, or energy? Any constraints?”</em> 🧐
    </p>
    <p style="font-size: 16px; line-height: 1.6;">
        Now try this instead:  
        <em>“Analyze the HVAC system and minimize energy consumption while keeping temperatures between 22-24°C. Provide a cost breakdown.”</em>  
    </p>
    <p style="font-size: 16px; line-height: 1.6;">
        That’s not just a prompt—it’s a <strong>clear strategy</strong> with defined objectives and boundaries.
        And that’s exactly what AI models need to perform at their best.
    </p>
</div>

<!-- Tip Section -->
<div style="background: #f5faff; padding: 20px; border-radius: 8px; border-left: 5px solid #0055d4; margin-top: 30px;">
    <h3 style="margin-top: 0; color: #0055d4;">💡 Pro Tip</h3>
    <p style="margin: 0; font-size: 16px; line-height: 1.6;">
        AI models appreciate well-structured instructions just like engineers appreciate complete design specs.
        Be specific, set clear goals, and watch the results improve!
    </p>
</div>

<!-- Upcoming Topics -->
<div style="margin-top: 40px; text-align: center;">
    <h3 style="color: #001a70;">What’s Ahead</h3>
    <ul style="list-style: none; padding: 0; font-size: 16px; line-height: 1.8;">
        <li>📚 Basic Prompting Types</li>
        <li>🧩 Advanced Strategies</li>
        <li>📊 Application-Specific Techniques</li>
    </ul>
    <p style="font-size: 16px; color: #333;">Let’s engineer some powerful AI conversations! 🛠️</p>
</div>


<!-- Section Header -->
<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 25px; border-radius: 12px; text-align: center; box-shadow: 0 4px 12px rgba(0,0,0,0.1);">
    <h1 style="margin-bottom: 10px; font-size: 30px;">📚 Basic Prompting Types</h1>
</div>

<!-- Spacer -->
<div style="height: 25px;"></div>

<!-- Zero-Shot Prompting -->
<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-bottom: 20px;">
    <h3 style="margin-top: 0; color: #001a70;">1️⃣ Zero-Shot Prompting</h3>
    <p style="font-size: 16px; line-height: 1.6;">
        Provide only the task without any examples.  
        <strong>Use When:</strong> The task is simple and well-known by the model.  
        <em>Example:</em> “Translate 'Hello' to French.”
    </p>
</div>


In [ ]:
# ==========================================================
# 1. Basic prompting: pick the model for the demos
# ==========================================================
# Your key + client were already loaded in the API check cell above via
# setup_openai(). Here we simply pick which model the basic-prompting demos use.
from google.colab import userdata
import openai
import os

# Reuse the same Colab secret the whole course uses (setup_openai read this too)
api_key = userdata.get('OPENAI_API_KEY')
if api_key is None:
    raise ValueError("API Key not found. Please store OPENAI_API_KEY in Colab secrets.")

os.environ["OPENAI_API_KEY"] = api_key
client = openai.OpenAI(api_key=api_key)   # OpenAI client for the create() calls below
print("OpenAI API key loaded and client ready.")

# Default model for the simple zero/one/few-shot demos (fast + cheap)
model_name = DEFAULT_MINI_MODEL
print(f"LLM model set to: {model_name}")

**What "temperature" means.** Temperature controls how random the model's word choices are. At `temperature=0` the model is (nearly) deterministic and picks its single most likely continuation, so the same prompt gives the same answer. Higher values (`0.7`, `1.0`) let it explore less-likely words, producing more varied and creative output. The demos below use `0` for reproducible results, and later switch to `0.7` on purpose to generate *different* reasoning paths for self-consistency.

In [ ]:
# ==========================================================
# 1. Zero-shot: ask with no examples (hidden formula sequence)
# ==========================================================
from openai import OpenAI
client = OpenAI()

# Zero-shot = we give ONLY the task, no worked example, and see if the model can
# infer the hidden rule behind the sequence on its own.
hard_sequence_prompt_zero = (
    "The sequence is: 3, 12, 27, 48, 75, ___. What’s next?"
)

response_zero_hard = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL,
    messages=[{"role": "user", "content": hard_sequence_prompt_zero}],
    temperature=0                       # temperature 0 = most deterministic answer
)

print("🔹 LLM Response (Zero-Shot - Hard Sequence):\n")
pretty_print(response_zero_hard.choices[0].message.content.strip(), title="🤖 Model Response")


<!-- One-Shot Prompting -->
<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-bottom: 20px;">
    <h3 style="margin-top: 0; color: #001a70;">2️⃣ One-Shot Prompting</h3>
    <p style="font-size: 16px; line-height: 1.6;">
        Provide one clear example along with the instruction.  
        <strong>Use When:</strong> You want to guide the model’s behavior with a single example.  
        <em>Example:</em> “Translate 'Hello' to French: Bonjour. Now translate 'Goodbye'.”
    </p>
</div>


In [ ]:
# ==========================================================
# 2. One-shot: add a single worked example
# ==========================================================
from openai import OpenAI
client = OpenAI()

model_name = DEFAULT_MINI_MODEL

# Zero-Shot Prompt (no example) -- the baseline to compare against
zero_shot_prompt = (
    "The sequence is: 1, 4, 2, 9, 3, 16, 4, ___. What number should replace the blank?"
)

# One-Shot Prompt = one solved example, THEN the new question in the same format
one_shot_prompt = (
    "Example:\n"
    "The sequence is: 1, 1, 2, 4, 3, 9, ___. What’s next?\n"
    "Answer: 4.\n\n"
    "Now solve this one:\n"
    "The sequence is: 1, 4, 2, 9, 3, 16, 4, ___. What number should replace the blank?"
)

# Run both prompts on the same model so the ONLY difference is the example
response_zero = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": zero_shot_prompt}],
    temperature=0
)

response_one = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": one_shot_prompt}],
    temperature=0
)

# Display Results side by side
print("🔹 Zero-Shot Response:\n" + "-"*40)
pretty_print(response_zero.choices[0].message.content.strip(), title="🤖 Model Response")

print("\n\n🔹 One-Shot Response:\n" + "-"*40)
pretty_print(response_one.choices[0].message.content.strip(), title="🤖 Model Response")


<!-- Few-Shot Prompting -->
<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-bottom: 20px;">
    <h3 style="margin-top: 0; color: #001a70;">3️⃣ Few-Shot Prompting</h3>
    <p style="font-size: 16px; line-height: 1.6;">
        Provide multiple examples to clearly demonstrate the pattern.  
        <strong>Use When:</strong> The task is complex or requires understanding a specific format.  
        <em>Example:</em>  
        - “Translate 'Hello' to French: Bonjour.”  
        - “Translate 'Goodbye' to French: Au revoir.”  
        - “Translate 'Thank you' to French: Merci.”  
        Now translate 'Good night'.
    </p>
</div>

<!-- Spacer -->
<div style="height: 30px;"></div>

<!-- Closing Tip -->
<div style="background: #f5faff; padding: 20px; border-radius: 8px; border-left: 5px solid #0055d4;">
    <h3 style="margin-top: 0; color: #0055d4;">💡 Quick Reminder</h3>
    <p style="margin: 0; font-size: 16px; line-height: 1.6;">
        The more complex the task, the more examples you should provide. But remember, too many examples can make prompts bulky and inefficient.
    </p>
</div>



In [ ]:
# ==========================================================
# 3. Few-shot: several examples for a harder pattern
# ==========================================================
from openai import OpenAI
client = OpenAI()

model_name = DEFAULT_CHAT_MODEL  # stronger model, since this pattern is harder

# Few-shot = multiple worked examples so the model can infer a multi-rule pattern
few_shot_prompt = (
    "Example 1:\n"
    "The sequence is: 1, 1, 2, 4, 3, 9, ___. What’s next?\n"
    "Answer: 4.\n\n"
    "Example 2:\n"
    "The sequence is: 1, 1, 2, 4, 4, 9, 7, 16, ___. What’s next?\n"
    "Answer: 11.\n\n"
    "Now try this one:\n"
    "The sequence is: 1, 1, 2, 4, 4, 9, 7, 16, 11, ___, 16, 36. What number should replace the blank?"
)

response_few = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": few_shot_prompt}],
    temperature=0
)

# Display Result
print("🔹 Few-Shot Prompting (Two Examples Provided):")
print("-" * 40)
pretty_print(response_few.choices[0].message.content.strip(), title="🤖 Model Response")

## 🧠 Advanced Prompting Techniques  

Moving beyond basic prompting methods like zero-shot and few-shot, advanced strategies help enhance the reasoning and adaptability of large language models (LLMs). These techniques guide the model's thought process to handle complex tasks more effectively.

---

### 🔗 Chain-of-Thought (CoT) Prompting  

Chain-of-Thought prompting encourages models to **explain their intermediate reasoning steps**, leading to more transparent and accurate conclusions. By structuring prompts to include logical steps, CoT improves the model’s ability to solve complex reasoning tasks.

**Why is CoT Important?**  
- ✔️ Improves performance on multi-step reasoning tasks.  
- ✔️ Helps produce logically structured and coherent responses.  
- ✔️ Breaks down complex problems into manageable steps.

📖 **Reference:** [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903)

---

*Next, explore practical examples of Chain-of-Thought prompting.*


In [ ]:
# ==========================================================
# 4. Chain-of-Thought: force step-by-step reasoning
# ==========================================================
model_name = DEFAULT_CHAT_MODEL

# A classic reasoning trap: the intuitive answer (100) is wrong; the answer is 5.
problem = (
    "If it takes 5 machines 5 minutes to make 5 widgets, "
    "how long would it take 100 machines to make 100 widgets?"
)

# Zero-Shot: just ask for the answer, no reasoning requested
zero_shot_prompt = problem + " Answer with just the number of minutes."

# Chain-of-Thought: ask the model to reason step by step first, then answer
cot_prompt = problem + " Let's think step by step, then give the final answer."

# Run both on the SAME problem to compare reasoning vs. no-reasoning
response_zero = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": zero_shot_prompt}],
    temperature=0,
)
response_cot = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": cot_prompt}],
    temperature=0,
)

# Display both so you can compare
print("Zero-Shot (no reasoning encouraged):\n" + "-" * 50)
pretty_print(response_zero.choices[0].message.content.strip(), title="Model Response")

print("\nChain-of-Thought (reasoning encouraged):\n" + "-" * 50)
pretty_print(response_cot.choices[0].message.content.strip(), title="Model Response")

# ✋ Hands-On Experiment: Observations  

📌 **Instructions:**  
- Run your experiments by changing the model type (e.g., `gpt-4.1-mini`, `gpt-4.1`, `o3`), temperature, and prompt style.  
- You can **either attach a screenshot/image of your results** or **write a brief summary of your observations (max half a page)**.

---

- **Model Used:**  
  _[Enter the model name you tried, e.g., gpt-4.1-mini, gpt-4.1, or o3]_

- **Temperature Setting:**  
  _[Enter the temperature you used, e.g., 0.0, 0.5, 0.7]_

- **Zero-Shot Result:**  
  _[Did Zero-Shot solve the problem correctly? Yes/No. Add a short explanation or attach an image.]_

- **Chain-of-Thought Result:**  
  _[Did Chain-of-Thought solve the problem better? Yes/No. Add a short explanation or attach an image.]_

- **Key Takeaways (Max Half Page or Screenshot):**  
  _[Summarize what you observed. Did a specific model perform better? How did temperature affect the results? What worked best? Attach image or write here.]_

---

✍️ *Try at least two models and different temperatures. Compare the results and reflect on how prompting strategies influence performance!*

## 🔁 Self-Consistency Prompting

While Chain-of-Thought (CoT) improves reasoning by encouraging step-by-step thinking, it may still produce **inconsistent or incorrect** answers, especially in complex scenarios.  
**Self-Consistency Prompting** enhances CoT by asking the model to **generate multiple reasoning paths** and then select the most common or consistent final answer.

### Why is Self-Consistency Useful?

- ✅ Reduces random reasoning errors.
- ✅ Boosts reliability on ambiguous or multi-path problems.
- ✅ Often improves performance on mathematical, logical, and symbolic tasks.

📖 **Reference**: [Self-Consistency Improves Chain of Thought Reasoning in Language Models](https://arxiv.org/abs/2203.11171)

---

*Next, we’ll see how Self-Consistency works in action using a complex reasoning example.*


In [ ]:
# ==========================================================
# 5. Self-consistency: sample many paths, take the majority
# ==========================================================
model_name = DEFAULT_CHAT_MODEL  # stronger model for multi-step reasoning

# The reasoning problem we will attack
problem_prompt = (
    "If a train travels at 60 miles per hour and leaves at 2 PM, and another train leaves "
    "the same station at 3 PM traveling at 90 miles per hour, when will the second train catch up to the first?"
)
cot_prompt = "Let's solve this step by step.\n" + problem_prompt

def run_self_consistency(prompt, num_attempts=3):
    """Sample the model several times at nonzero temperature to get varied
    reasoning paths, so we can compare them and take the majority."""
    answers = []
    for _ in range(num_attempts):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,          # nonzero temp = a different reasoning path each run
        )
        answers.append(response.choices[0].message.content.strip())
    return answers

# Baseline: a single deterministic CoT attempt
response_cot = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": cot_prompt}],
    temperature=0,
)
cot_answer = response_cot.choices[0].message.content.strip()

# Self-consistency: several attempts, then take the most common answer
sc_answers = run_self_consistency(cot_prompt, num_attempts=3)
from collections import Counter
most_common_answer = Counter(sc_answers).most_common(1)[0]

# Display everything with pretty_print, same as the other cells
pretty_print(cot_answer, title="Single CoT Attempt")
for idx, ans in enumerate(sc_answers, 1):
    pretty_print(ans, title=f"Self-Consistency Attempt {idx}")
pretty_print(
    f"Most common answer appeared {most_common_answer[1]} of {len(sc_answers)} times:\n\n{most_common_answer[0]}",
    title="Self-Consistency: Selected Answer",
)

> **Pause and think.** Did the single Chain-of-Thought answer match the self-consistency majority? When five samples *disagree*, which one would you trust, and why does voting across paths help on hard reasoning problems?

**Your notes** *(double-click to edit)*

- CoT single answer vs. majority: 
- Did the samples agree? 
- When would you trust voting over a single pass: 

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 25px; border-radius: 12px; text-align: center;">
    <h1 style="margin-bottom: 10px;">📚 Exploring More Advanced Prompting Strategies</h1>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-top: 20px;">
    <ul style="font-size: 16px; line-height: 1.8;">
        <li><strong>🧩 Tree-of-Thought (ToT) Prompting:</strong> Explores multiple reasoning paths like a decision tree, helping the model evaluate and compare various solutions before choosing the best one.</li>
        <li><strong>🤖 ReAct (Reasoning and Acting) Prompting:</strong> Combines reasoning steps with actions, including API calls or external tool usage. Ideal for interactive agents and dynamic decision-making tasks.</li>
        <li><strong>🔄 Reflexion Prompting:</strong> Encourages the model to critique its own responses and iteratively improve them, simulating self-correction and learning.</li>
    </ul>
</div>

<div style="margin-top: 40px; text-align: center;">
    <h2 style="color: #001a70;">✋ Hands-On Task: Compare Prompting Strategies</h2>
</div>

<div style="background: #f5faff; padding: 20px; border-radius: 8px; border-left: 5px solid #0055d4;">
    <p style="font-size: 16px;">
        📌 <strong>Task Instructions:</strong><br>
        - Experiment with <strong>Self-Consistency</strong>, <strong>Tree-of-Thought</strong>, and <strong>ReAct</strong> prompting methods.<br>
        - Try to solve the following problem using each method and compare the results.
    </p>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-top: 20px;">
    <h3>🧠 <strong>Challenge Problem:</strong></h3>
    <p style="font-size: 16px;">A farmer has chickens and rabbits in a cage. There are 35 heads and 94 legs. How many chickens and rabbits are there?</p>
</div>

<div style="margin-top: 40px;">
    <ul style="font-size: 16px; line-height: 1.8;">
        <li>Try different models (e.g., <code>gpt-4.1-mini</code>, <code>gpt-4.1</code>, <code>o3</code>).</li>
        <li>Experiment with different temperatures (e.g., <code>0.0</code>, <code>0.5</code>, <code>0.7</code>).</li>
        <li>Use both direct prompts and advanced strategies like CoT, Self-Consistency, or ReAct.</li>
    </ul>
</div>

<div style="margin-top: 40px; text-align: center;">
    <h2 style="color: #001a70;">📖 Observations</h2>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4;">
    <ul style="font-size: 16px; line-height: 1.8;">
        <li><strong>Model and Strategy Used:</strong><br>_[Enter the model and prompting strategy you tried]_</li>
        <li><strong>Was the Correct Answer Found?</strong><br>_[Yes/No. Explain briefly or attach a screenshot]_</li>
        <li><strong>Key Takeaways (Max Half Page or Screenshot):</strong><br>_[Summarize how different strategies performed. What worked best? Why?]_</li>
    </ul>
</div>

<div style="margin-top: 20px; text-align: center;">
    ✍️ <em>Hint: Try breaking down the problem into equations or ask the model to explain its steps before giving the final answer. Notice which strategies lead to faster and more accurate results!</em>
</div>

In [ ]:
# ==========================================================
# 6. Your turn: try different strategies, models, temperatures
# ==========================================================
# 📝 Instructions:
# - Change 'model_name' to try different models (e.g., DEFAULT_MINI_MODEL, DEFAULT_CHAT_MODEL, "o3").
# - Adjust 'temperature' to test how creativity affects reasoning.
# - Try Self-Consistency by sampling multiple outputs and comparing answers.
# - Optionally, explore Tree-of-Thought and ReAct patterns by modifying prompts.
# ✅ Your Experiment Starts Here 👇

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 25px; border-radius: 12px; text-align: center;">
    <h1 style="margin-bottom: 10px;">📌 Conclusion</h1>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-top: 20px;">
    <p style="font-size: 16px; line-height: 1.8;">
        In this hands-on exploration, different advanced prompting strategies were tested to solve reasoning-based challenges.
        Through experimenting with <strong>Chain-of-Thought (CoT)</strong>, <strong>Self-Consistency</strong>, and other methods,
        the following key insights were observed:
    </p>
    <ul style="font-size: 16px; line-height: 1.8;">
        <li>Advanced prompting techniques significantly improve model performance, especially on complex, multi-step problems.</li>
        <li>Changing the <strong>model type</strong> and <strong>temperature</strong> can drastically affect reasoning quality and creativity.</li>
        <li>Some strategies, like <strong>Self-Consistency</strong>, help reduce random errors by exploring multiple reasoning paths.</li>
        <li>For ambiguous or challenging problems, combining strategies (e.g., CoT + Self-Consistency) often leads to the most reliable results.</li>
    </ul>
</div>

<div style="background: #f5faff; padding: 20px; border-radius: 8px; border-left: 5px solid #0055d4; margin-top: 20px;">
    <p style="font-size: 16px; font-style: italic;">
        📖 <em>Remember: Prompt engineering is both an art and a science. The more you experiment, the better you understand how to guide LLMs effectively!</em>
    </p>
</div>

<div style="margin-top: 40px; text-align: center;">
    <h3 style="color: #001a70;">✍️ Final Reflection</h3>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4;">
    <p style="font-size: 16px;">
        _[Write 2-3 sentences summarizing what you personally learned about prompting strategies and how model selection or temperature influenced the results.]_
    </p>
</div>
